## POT RF, In-Situ, Time Varying

In this section, 
1. Time varying thresholds, POT of RF, and associated NTR and RD in a 2.5 days windows are achieved.

In [1]:
import pandas as pd
import numpy as np
import glob
from datetime import timedelta
import os
from pathlib import Path

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 5

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    
    base_path = core_directory / Path(f'{loc}')
    # ────────────────────────────────────────────────────────────────────────────────
    # 📌 Load rainfall and NTR datasets
    # ────────────────────────────────────────────────────────────────────────────────
    RF_File = rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv'
    NTR_File = rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv'
    RD_File = rf'{dataset_directory}/In_Situ/usgs_discharge.csv'

    df_rf = pd.read_csv(RF_File)
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index(time_col_rf)
    
    df_ntr = pd.read_csv(NTR_File)
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index(time_col_ntr)
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    rainfall_series = df_rf[acc_hr].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/TimeVarying'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in rainfall_series.groupby(rainfall_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == events_per_year:
                break
    
        if len(selected) < events_per_year:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [rainfall_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            ntr_window = df_ntr.loc[start:end]
            max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
            all_pot.append({'time': t, 'POT_RF': val, 'Max_NTR_in_5d': max_ntr, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_RF_TimeVarying_{events_per_year}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'RF_TimeVarying_Thresholds_{events_per_year}.csv'), index=False)
    

    df_pot = pd.read_csv(rf'{output_dir}/POT_RF_TimeVarying_{events_per_year}.csv')
    df_pot = df_pot.dropna()
    df_pot = df_pot.drop(columns=['Year'])
    
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
    
    df_rd = pd.read_csv(RD_File)
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
              
    # Ensure RD is daily (rounded to date)
    df_rd.index = df_rd.index.date  
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        event_date = event_time.date()
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_rd.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot['date'] = df_pot.index.date  
    df_pot = df_pot.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    # merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}/POT_RF_TimeVarying_NTR_RD_{events_per_year}.csv',index=False)                
                    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\241837000.py:130: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\241837000.py:130: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')


## POT RF, In-Situ, Stationary

In this section, 
1. Constant threshold, POT of RF, and associated NTR and RD in a 2.5 days windows are achieved.

In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta
import os
from pathlib import Path

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 5

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:

    base_path = core_directory / Path(f'{loc}')

    RF_File = rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv'
    NTR_File = rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv'
    RD_File = rf'{dataset_directory}/In_Situ/usgs_discharge.csv'

    df_rf = pd.read_csv(RF_File)
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')
    
    df_ntr = pd.read_csv(NTR_File)
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index('time')
    
    rainfall_series = df_rf[acc_hr].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/Stationary'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Find constant threshold for 5 events/year
    # ────────────────────────────────────────────────────────────────────────────────
    n_years = rainfall_series.index.year.nunique()
    target_events = events_per_year * n_years
    
    def count_events(threshold):
        selected = []
        for t, val in rainfall_series.items():
            if val >= threshold:
                if any(abs((t - sel).days) < decluster_window.days for sel in selected):
                    continue
                selected.append(t)
        return len(selected)
    
    # Start from the initial threshold (top N values)
    sorted_values = rainfall_series.sort_values(ascending=False)
    low, high = sorted_values.iloc[-1], sorted_values.iloc[0]  
    best_threshold = None
    
    # Binary search for the threshold that gives ~ target_events
    for _ in range(50):  # 50 iterations for convergence
        mid = (low + high) / 2
        events = count_events(mid)
        if events > target_events:
            low = mid
        else:
            high = mid
        best_threshold = mid
    
    threshold_value = best_threshold
    
    # Declustering window: ±2.5 days
    decluster_window = timedelta(days=2.5)
    
    # Step 1: Identify POT events (≥ threshold_value) with 2.5-day declustering
    pot_selected = []
    for t, val in rainfall_series.sort_values(ascending=False).items():
        if val >= threshold_value:
            if any(abs((t - sel).days) <= decluster_window.days for sel in pot_selected):
                continue
            pot_selected.append(t)
    
    # Step 2: Mark all times within ±2.5 days of any POT event
    pot_exclusion_periods = [(event_time - decluster_window, event_time + decluster_window)
                             for event_time in pot_selected]
    
    def is_in_exclusion_period(time, exclusion_periods):
        return any(start <= time <= end for start, end in exclusion_periods)
    
    # Step 3: Filter below threshold and outside exclusion windows
    below_threshold_times = df_rf[
        (df_rf[acc_hr] < threshold_value) &
        (~df_rf.index.map(lambda t: is_in_exclusion_period(t, pot_exclusion_periods)))
    ].index
    
    # Step 4: For each POT event, find max NTR in ±2.5 days
    pot_list = []
    for t in pot_selected:
        ntr_window = df_ntr.loc[t - decluster_window : t + decluster_window]
        max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
        pot_list.append({
            'time': t,
            'POT_RF': df_rf.loc[t, acc_hr],
            'Max_NTR_in_5d': max_ntr
        })
    
    df_pot_events = pd.DataFrame(pot_list).sort_values('time').reset_index(drop=True)
    
    output_below_file = os.path.join(output_dir, f'RF_Stationary_Threshold_{round(threshold_value,2)}_{events_per_year}.csv')
    df_pot_events.to_csv(output_below_file, index=False)
    
    output_below_file = os.path.join(output_dir, f'POT_RF_Stationary_{events_per_year}.csv')
    df_pot_events.to_csv(output_below_file, index=False)
    
    
    df_pot = pd.read_csv(rf'{output_dir}/POT_RF_Stationary_{events_per_year}.csv')
    df_pot = df_pot.dropna()
    
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
    
    df_rd = pd.read_csv(RD_File)
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
              
    # Ensure RD is daily (rounded to date)
    df_rd.index = df_rd.index.date  
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        event_date = event_time.date()  
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_rd.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot['date'] = df_pot.index.date  
    df_pot = df_pot.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}/POT_RF_Stationary_NTR_RD_{events_per_year}.csv',index=False)                
                                  

C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\3432595040.py:135: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\3432595040.py:135: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')


## POT RF, Reanalysis, Time Varying

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 10

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    # ────────────────────────────────────────────────────────────────────────────────
    # 📌 Load datasets
    # ────────────────────────────────────────────────────────────────────────────────
    df_rf = pd.read_csv(rf'{dataset_directory}\Reanalysis\APCP_Basin_Average_{loc}_Combined_accumulated.csv')
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index(time_col_rf)
    
    df_ntr = pd.read_csv(rf'{dataset_directory}\Reanalysis\Compare_Points_{loc}_Avg.csv')
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index(time_col_ntr)

    df_rd = pd.read_csv(rf'{dataset_directory}\Reanalysis\GloFAS_{loc}_Basin_Avg.csv')
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    rainfall_series = df_rf[acc_hr].dropna()
    decluster_window = timedelta(days=3)
    output_dir = rf'{core_directory}\{loc}\Reanalysis\TimeVarying'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in rainfall_series.groupby(rainfall_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == events_per_year:
                break
    
        if len(selected) < events_per_year:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [rainfall_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            ntr_window = df_ntr.loc[start:end]
            max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
            all_pot.append({'time': t, 'POT_RF': val, 'Max_NTR_in_5d': max_ntr, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_RF_TimeVarying_{events_per_year}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'RF_TimeVarying_Thresholds_{events_per_year}.csv'), index=False)
    
    
    df_pot = pd.read_csv(rf'{output_dir}\POT_RF_TimeVarying_{events_per_year}.csv')
    df_pot = df_pot.dropna()
    df_pot = df_pot.drop(columns=['Year'])
    
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
              
    # Ensure RD is daily (rounded to date)
    df_rd.index = df_rd.index.date  
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        event_date = event_time.date()  
    
        start_date = event_date - timedelta(days=3)
        end_date = event_date + timedelta(days=3)
    
        # Subset discharge data in that window
        window_data = df_rd.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot['date'] = df_pot.index.date  
    df_pot = df_pot.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}\POT_RF_TimeVarying_NTR_RD_{events_per_year}.csv',index=False)                
                    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\1433693776.py:124: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\1433693776.py:124: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')


## POT RF, Reanalysis, Stationary

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_rf = 'time'
time_col_ntr = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
events_per_year = 10

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:

    df_rf = pd.read_csv(rf'{dataset_directory}\Reanalysis\APCP_Basin_Average_{loc}_Combined_accumulated.csv')
    df_rf[time_col_rf] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index(time_col_rf)
    
    df_ntr = pd.read_csv(rf'{dataset_directory}\Reanalysis\Compare_Points_{loc}_Avg.csv')
    df_ntr[time_col_ntr] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index(time_col_ntr)

    df_rd = pd.read_csv(rf'{dataset_directory}\Reanalysis\GloFAS_{loc}_Basin_Avg.csv')
    df_rd['time'] = pd.to_datetime(df_rd[time_col_rd])
    df_rd = df_rd.drop(columns=[time_col_rd])
    df_rd = df_rd.set_index('time')
    
    rainfall_series = df_rf[acc_hr].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{core_directory}\{loc}\Reanalysis\Stationary'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Find constant threshold for 5 events/year
    # ────────────────────────────────────────────────────────────────────────────────
    n_years = rainfall_series.index.year.nunique()
    target_events = events_per_year * n_years
    
    def count_events(threshold):
        selected = []
        for t, val in rainfall_series.items():
            if val >= threshold:
                if any(abs((t - sel).days) < decluster_window.days for sel in selected):
                    continue
                selected.append(t)
        return len(selected)
    
    # Start from the initial threshold (top N values)
    sorted_values = rainfall_series.sort_values(ascending=False)
    low, high = sorted_values.iloc[-1], sorted_values.iloc[0]  # min, max rainfall
    best_threshold = None
    
    # Binary search for the threshold that gives ~ target_events
    for _ in range(50):  # 50 iterations for convergence
        mid = (low + high) / 2
        events = count_events(mid)
        if events > target_events:
            low = mid
        else:
            high = mid
        best_threshold = mid
    
    threshold_value = best_threshold

    # Declustering window: ±2.5 days
    decluster_window = timedelta(days=2.5)
    
    # Step 1: Identify POT events (≥ threshold_value) with 2.5-day declustering
    pot_selected = []
    for t, val in rainfall_series.sort_values(ascending=False).items():
        if val >= threshold_value:
            if any(abs((t - sel).days) <= decluster_window.days for sel in pot_selected):
                continue
            pot_selected.append(t)
    
    # Step 2: Mark all times within ±2.5 days of any POT event
    pot_exclusion_periods = [(event_time - decluster_window, event_time + decluster_window)
                             for event_time in pot_selected]
    
    def is_in_exclusion_period(time, exclusion_periods):
        return any(start <= time <= end for start, end in exclusion_periods)
    
    # Step 3: Filter below threshold and outside exclusion windows
    below_threshold_times = df_rf[
        (df_rf[acc_hr] < threshold_value) &
        (~df_rf.index.map(lambda t: is_in_exclusion_period(t, pot_exclusion_periods)))
    ].index
    
    # Step 4: For each POT event, find max NTR in ±2.5 days
    pot_list = []
    for t in pot_selected:
        ntr_window = df_ntr.loc[t - decluster_window : t + decluster_window]
        max_ntr = ntr_window[ntr_col].max() if not ntr_window.empty else np.nan
        pot_list.append({
            'time': t,
            'POT_RF': df_rf.loc[t, acc_hr],
            'Max_NTR_in_5d': max_ntr
        })
    
    df_pot_events = pd.DataFrame(pot_list).sort_values('time').reset_index(drop=True)
    
    output_below_file = os.path.join(output_dir, f'POT_RF_Stationary_Threshold_{round(threshold_value,2)}_{events_per_year}.csv')
    df_pot_events.to_csv(output_below_file, index=False)
    
    output_below_file = os.path.join(output_dir, f'POT_RF_Stationary_{events_per_year}.csv')
    df_pot_events.to_csv(output_below_file, index=False)


    df_pot = pd.read_csv(rf'{output_dir}\POT_RF_Stationary_{events_per_year}.csv')
    df_pot = df_pot.dropna()
    
    df_pot['time'] = pd.to_datetime(df_pot['time'])
    df_pot = df_pot.set_index('time')
    
    # Ensure RD is daily (rounded to date)
    df_rd.index = df_rd.index.date  

    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in df_pot.index:
        event_date = event_time.date()  
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_rd.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    df_pot.index = pd.to_datetime(df_pot.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    df_pot = df_pot.copy()
    df_pot['date'] = df_pot.index.date  
    df_pot = df_pot.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  
    
    # Step 3: Merge on 'date'
    merged_df = df_pot.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['time'] = pd.to_datetime(merged_df['time'])
                   
    merged_df.to_csv(rf'{output_dir}\POT_RF_Stationary_NTR_RD_{events_per_year}.csv',index=False)                
                                  

C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\677270952.py:129: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_44496\677270952.py:129: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pot.index = pd.to_datetime(df_pot.index).round('H')
